In [ ]:
import pandas as pd

df = pd.read_csv('../raw_data/cleaned_enriched_data.csv',
                 parse_dates=['founded_at', 'first_funding_at', 'last_funding_at'])
print(f"Loaded: {df.shape}")

Loaded: (44363, 23)


In [ ]:
# ============================================================
# FEATURE ENGINEERING — adapted for enriched dataset
# ============================================================
import numpy as np

df_fe = df.copy()

# --- Date parsing (safety check, should already be datetime from cleaning) ---
date_cols = ["founded_at", "first_funding_at", "last_funding_at"]
for col in date_cols:
    df_fe[col] = pd.to_datetime(df_fe[col], errors="coerce")

# --- avg_raised_per_round ---
df_fe["avg_raised_per_round"] = np.where(
    df_fe["funding_rounds"] > 0,
    df_fe["funding_total_usd"] / df_fe["funding_rounds"],
    np.nan
)

# --- age_first_funding_days ---
df_fe["age_first_funding_days"] = (
    df_fe["first_funding_at"] - df_fe["founded_at"]
).dt.days
df_fe["age_first_funding_days"] = df_fe["age_first_funding_days"].clip(lower=0)

# --- has_multiple_rounds ---
df_fe["has_multiple_rounds"] = (df_fe["funding_rounds"] > 1).astype(int)

# --- funding_span_days ---
df_fe["funding_span_days"] = (
    df_fe["last_funding_at"] - df_fe["first_funding_at"]
).dt.days
df_fe["funding_span_days"] = df_fe["funding_span_days"].clip(lower=0)

# --- avg_years_between_rounds ---
df_fe["avg_years_between_rounds"] = np.where(
    df_fe["funding_rounds"] > 1,
    (df_fe["funding_span_days"] / 365.25) / (df_fe["funding_rounds"] - 1),
    np.nan
)
median_gap = df_fe.loc[
    df_fe["funding_rounds"] > 1, "avg_years_between_rounds"
].median()
df_fe["avg_years_between_rounds"] = df_fe["avg_years_between_rounds"].fillna(median_gap)

# --- region_group (from country_code) ---
eu_uk = {
    "gbr","deu","fra","esp","ita","nld","bel","swe","dnk","fin","irl",
    "aut","prt","pol","cze","hun","grc","rou","bgr","hrv","svk","svn",
    "est","lva","ltu","lux","mlt","cyp","rom"
}
asia = {
    "chn","ind","jpn","kor","sgp","hkg","idn","mys","tha","vnm",
    "phl","twn","pak","bgd","lka","npl","isr","are","sau","kwt",
    "qat","omn","bhr"
}
americas = {
    "mex","bra","arg","chl","col","per","ury","ecu","bol","pry",
    "cri","pan","gtm","dom","jam","tto"
}

def map_region(country_code):
    if pd.isna(country_code) or country_code == "unknown":
        return "Unknown"
    if country_code == "usa":
        return "USA"
    if country_code == "can":
        return "Canada"
    if country_code in eu_uk:
        return "EU_UK"
    if country_code in asia:
        return "Asia"
    if country_code == "aus":
        return "Australia"
    if country_code in americas:
        return "Rest_Americas"
    return "Rest_World"

df_fe["region_group"] = df_fe["country_code"].apply(map_region)

# --- market_clean + industry_group ---
df_fe["market_clean"] = df_fe["market"].str.strip().str.lower()

def map_industry(market):
    if pd.isna(market) or market == "unknown":
        return "Unknown"
    market = market.lower()
    if market in ["software","enterprise software","analytics","web hosting",
                   "security","saas","cloud computing","big data","internet","technology"]:
        return "Software_Data"
    if market in ["curated web","social media","messaging","news","music","games",
                   "apps","video","entertainment","social network media","photography","search"]:
        return "Consumer_Internet"
    if market in ["biotechnology","health care","health and wellness","medical"]:
        return "Health_Bio"
    if market in ["e-commerce","marketplaces"]:
        return "Ecommerce"
    if market in ["consulting","advertising","public relations","sales and marketing","design"]:
        return "Services"
    if market in ["manufacturing","real estate","hospitality","travel","fashion",
                   "automotive","transportation","sports"]:
        return "Real_World"
    if market in ["hardware + software","semiconductors","networking"]:
        return "Hardware_DeepTech"
    if market == "clean technology":
        return "Energy"
    if market == "finance":
        return "FinTech"
    if market == "education":
        return "Education"
    return "Other"

df_fe["industry_group"] = df_fe["market_clean"].apply(map_industry)

# --- Summary ---
print(f"Shape: {df_fe.shape}")
print(f"\nNew features added:")
new_cols = ['avg_raised_per_round', 'age_first_funding_days', 'has_multiple_rounds',
            'funding_span_days', 'avg_years_between_rounds', 'region_group',
            'market_clean', 'industry_group']
for col in new_cols:
    nulls = df_fe[col].isna().sum()
    print(f"  {col:30} nulls: {nulls}")

print(f"\nRegion distribution:")
print(df_fe['region_group'].value_counts())
print(f"\nIndustry distribution:")
print(df_fe['industry_group'].value_counts())

Shape: (44363, 31)

New features added:
  avg_raised_per_round           nulls: 0
  age_first_funding_days         nulls: 0
  has_multiple_rounds            nulls: 0
  funding_span_days              nulls: 0
  avg_years_between_rounds       nulls: 0
  region_group                   nulls: 0
  market_clean                   nulls: 0
  industry_group                 nulls: 0

Region distribution:
region_group
USA              25484
EU_UK             6777
Unknown           4916
Asia              3629
Canada            1244
Rest_World        1158
Rest_Americas      864
Australia          291
Name: count, dtype: int64

Industry distribution:
industry_group
Other                12234
Software_Data         7296
Consumer_Internet     5694
Health_Bio            5239
Unknown               3493
Real_World            2664
Ecommerce             1877
Services              1756
Hardware_DeepTech     1471
Energy                1083
FinTech                788
Education              768
Name: count, dty

In [4]:
df_fe.to_csv('../raw_data/features_enriched_data.csv', index=False)
print(f"Saved: {df_fe.shape}")

Saved: (44363, 31)
